In [2]:
from pyspark.sql import SparkSession
import os

try:
    spark.stop()
except:
    pass

import socket
driver_host = socket.gethostbyname('jupyter_pyspark')

spark = SparkSession.builder \
    .appName("Lab2_Final_Step") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", driver_host) \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.driver.port", "0") \
    .config("spark.driver.blockManager.port", "0") \
    .config("spark.executor.extraClassPath", "/opt/spark/drivers/*") \
    .config("spark.driver.extraClassPath", "/opt/spark/drivers/*") \
    .config("spark.jars", "/opt/spark/drivers/postgresql-42.7.1.jar") \
    .config("spark.sql.adaptive.enabled", "false") \
    .getOrCreate()

print(f"Spark Session created successfully!")
print(f"Spark Master: {spark.sparkContext.master}")
print(f"Spark App ID: {spark.sparkContext.applicationId}")

db_url = "jdbc:postgresql://db:5432/lab_db"
db_params = {
    "user": "postgres", 
    "password": "password", 
    "driver": "org.postgresql.Driver"
}

try:
    df = spark.read.jdbc(url=db_url, table="mock_data", properties=db_params)
    
    
    count = df.count()
    print(f"ПОБЕДА! СТРОК В БАЗЕ: {count}")
    df.show(5)
except Exception as e:
    print(f"Поражение! Ошибка при чтении данных: {e}")
    
print(f"\nКоличество воркеров в кластере: {len(spark.sparkContext._jsc.sc().statusTracker().getExecutorInfos())}")

Spark Session created successfully!
Spark Master: spark://spark-master:7077
Spark App ID: app-20260325205920-0000
ПОБЕДА! СТРОК В БАЗЕ: 10000
+--------------------+-------------------+------------------+--------------------+--------------------+----------------+--------------------+-----------------+-----------------+------------------+-----------------+----------------+--------------------+--------------+------------------+------------+----------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------+----------+-----------+-------------+------------+--------------------+------------+--------------------+-------------+------------+-------------+----------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------+----------------+--------------------+--------------+----------------+-------

In [3]:
from pyspark import SparkContext

sc = spark.sparkContext

executor_info = sc._jsc.sc().statusTracker().getExecutorInfos()

print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"Spark Master: {spark.sparkContext.master}")
print(f"Количество активных задач (параллелизм по умолчанию): {spark.sparkContext.defaultParallelism}")

print(f"\nАктивные исполнители: {len(executor_info)}")
for i, executor in enumerate(executor_info):
    executor_str = executor.toString()
    print(f"  - Executor {i}: {executor_str}")

Application ID: app-20260325205920-0000
Spark Master: spark://spark-master:7077
Количество активных задач (параллелизм по умолчанию): 2

Активные исполнители: 2
  - Executor 0: org.apache.spark.SparkExecutorInfoImpl@3b3fe9d5
  - Executor 1: org.apache.spark.SparkExecutorInfoImpl@350f2379


In [4]:
from pyspark.sql import functions as F

print("Категории товаров")
df.select("product_category").distinct().show()

unique_stores = df.select('store_name').distinct().count()
print(f"Количество уникальных магазинов: {unique_stores} ---")


print("Проверка на NULL (цена и дата)")
df.select(
    F.count(F.when(F.col("product_price").isNull(), 1)).alias("null_prices"),
    F.count(F.when(F.col("sale_date").isNull(), 1)).alias("null_dates")
).show()

print("Топ-5 стран по количеству покупателей")
df.groupBy("customer_country").count().orderBy(F.desc("count")).show(5)

print("Средняя цена товара по категориям питомцев")
df.groupBy("pet_category").agg(F.round(F.avg("product_price"), 2).alias("avg_price")).show()

--- Категории товаров ---
+----------------+
|product_category|
+----------------+
|            Cage|
|            Food|
|             Toy|
+----------------+

--- Количество уникальных магазинов: 383 ---
--- Проверка на NULL (цена и дата) ---
+-----------+----------+
|null_prices|null_dates|
+-----------+----------+
|          0|         0|
+-----------+----------+

--- Топ-5 стран по количеству покупателей ---
+----------------+-----+
|customer_country|count|
+----------------+-----+
|           China| 1738|
|       Indonesia| 1174|
|          Russia|  628|
|     Philippines|  555|
|          Brazil|  385|
+----------------+-----+
only showing top 5 rows

--- Средняя цена товара по категориям питомцев ---
+------------+---------+
|pet_category|avg_price|
+------------+---------+
|    Reptiles|    49.95|
|        Fish|    51.14|
|       Birds|    50.73|
|        Dogs|    51.66|
|        Cats|    50.54|
+------------+---------+



In [5]:
from pyspark.sql import functions as F

# dim_suppliers
dim_suppliers = df.select(
    "supplier_name", "supplier_contact", "supplier_email", 
    "supplier_phone", "supplier_address", "supplier_city", "supplier_country"
).dropDuplicates(["supplier_name"]) \
 .withColumn("supplier_id", F.monotonically_increasing_id())

# dim_stores
dim_stores = df.select(
    "store_name", "store_location", "store_city", 
    "store_state", "store_country", "store_phone", "store_email"
).dropDuplicates(["store_name"]) \
 .withColumn("store_id", F.monotonically_increasing_id())

# dim_customers
dim_customers = df.select(
    "customer_first_name", "customer_last_name", "customer_email", 
    "customer_age", "customer_country", "customer_postal_code"
).dropDuplicates(["customer_email"]) \
 .withColumn("customer_id", F.monotonically_increasing_id())

# dim_sellers (JOIN с магазинами)
dim_sellers = df.join(dim_stores, "store_name") \
    .select("seller_first_name", "seller_last_name", "seller_email", "store_id") \
    .dropDuplicates(["seller_email"]) \
    .withColumn("seller_id", F.monotonically_increasing_id())

# dim_products (JOIN с поставщиками)
dim_products = df.join(dim_suppliers, "supplier_name") \
    .select("product_name", "product_category", "product_brand", "product_price", "supplier_id") \
    .dropDuplicates(["product_name"]) \
    .withColumn("product_id", F.monotonically_increasing_id())

# dim_pets
dim_pets = df.join(dim_customers, "customer_email") \
    .select("customer_pet_name", "customer_pet_type", "customer_pet_breed", "customer_id", "customer_email") \
    .dropDuplicates(["customer_pet_name", "customer_email"]) \
    .withColumn("pet_id", F.monotonically_increasing_id()) \
    .drop("customer_email")

# fact_sales
fact_sales = df.alias("raw") \
    .join(dim_customers.alias("c"), "customer_email") \
    .join(dim_sellers.alias("sel"), "seller_email") \
    .join(dim_products.alias("p"), "product_name") \
    .join(dim_stores.alias("st"), "store_name") \
    .select(
        F.col("raw.sale_date").cast("date"),
        F.col("c.customer_id"),
        F.col("sel.seller_id"),
        F.col("p.product_id"),
        F.col("st.store_id"),
        F.col("raw.sale_quantity").cast("int"),
        F.col("raw.sale_total_price").cast("decimal(18,2)")
    )

print("ЗаписываЕЕм PostgreSQL...")

tables = {
    "dim_suppliers": dim_suppliers,
    "dim_stores": dim_stores,
    "dim_customers": dim_customers,
    "dim_sellers": dim_sellers,
    "dim_products": dim_products,
    "dim_pets": dim_pets,
    "fact_sales": fact_sales
}

for name, table_df in tables.items():
    table_df.write.jdbc(url=db_url, table=name, mode="overwrite", properties=db_params)
    print(f"Table {name} is written.")

print("Все таблицы Снежинки созданы в Postgres!")

Начинаю запись в PostgreSQL...
Table dim_suppliers is written.
Table dim_stores is written.
Table dim_customers is written.
Table dim_sellers is written.
Table dim_products is written.
Table dim_pets is written.
Table fact_sales is written.
--- Все таблицы Снежинки созданы в Postgres! ---


In [6]:
# Просто проверка, чтобы не ошибиться
print("Колонки фактов:", spark.read.jdbc(url=db_url, table="fact_sales", properties=db_params).columns)
print("Колонки продуктов:", spark.read.jdbc(url=db_url, table="dim_products", properties=db_params).columns)

Колонки фактов: ['sale_date', 'customer_id', 'seller_id', 'product_id', 'store_id', 'sale_quantity', 'sale_total_price']
Колонки продуктов: ['product_name', 'product_category', 'product_brand', 'product_price', 'supplier_id', 'product_id']


In [12]:
from pyspark.sql import functions as F
import requests
import json

db_url = "jdbc:postgresql://db:5432/lab_db"
db_params = {"user": "postgres", "password": "password", "driver": "org.postgresql.Driver"}

df_fact = spark.read.jdbc(url=db_url, table="fact_sales", properties=db_params)
df_prod = spark.read.jdbc(url=db_url, table="dim_products", properties=db_params)

df_joined = df_fact.join(df_prod, "product_id")

report_products = df_joined.groupBy("product_name", "product_category") \
    .agg(
        F.sum("sale_quantity").alias("total_sales_quantity"),
        F.sum("sale_total_price").alias("total_revenue"),
        F.round(F.avg("product_price"), 2).alias("avg_product_price")
    ) \
    .orderBy(F.desc("total_sales_quantity")) \
    .limit(10)

data_to_insert = report_products.collect()

clickhouse_url = "http://clickhouse_db:8123/"
auth_params = {"user": "user", "password": "password"}

create_table_query = """
CREATE TABLE IF NOT EXISTS reports.report_product_sales (
    product_name String,
    product_category String,
    total_sales_quantity Int64,
    total_revenue Float64,
    avg_product_price Float64
) ENGINE = MergeTree()
ORDER BY product_name
"""

try:

    response = requests.post(
        clickhouse_url,
        params=auth_params,
        data=create_table_query
    )
    print(f"Создание таблицы: {response.status_code} - {response.text}")
    
    insert_template = "INSERT INTO reports.report_product_sales (product_name, product_category, total_sales_quantity, total_revenue, avg_product_price) VALUES"
    
    for row in data_to_insert:
        insert_query = f"""
        INSERT INTO reports.report_product_sales 
        (product_name, product_category, total_sales_quantity, total_revenue, avg_product_price) 
        VALUES ('{row.product_name}', '{row.product_category}', {row.total_sales_quantity}, {row.total_revenue}, {row.avg_product_price})
        """
        response = requests.post(
            clickhouse_url,
            params=auth_params,
            data=insert_query
        )
        if response.status_code != 200:
            print(f"Ошибка при вставке {row.product_name}: {response.text}")
    
    print("Данные УсПеШнО записаны в ClickHouse!")
    
    check_query = "SELECT * FROM reports.report_product_sales"
    response = requests.post(
        clickhouse_url,
        params=auth_params,
        data=check_query
    )
    print("\nДанные в ClickHouse:")
    print(response.text)
    
except Exception as e:
    print(f"Ошибка: {e}")

Создание таблицы: 200 - 
✅ Данные успешно записаны в ClickHouse!

Данные в ClickHouse:
Cat Toy	Toy	18120	833806.88	62.17
Dog Food	Toy	18298	848567.19	21.37
Bird Cage	Toy	18205	847478.05	12.55



In [17]:
report_customers = df_fact.join(df_customers, "customer_id") \
    .groupBy("customer_first_name", "customer_last_name", "customer_email", "customer_country") \
    .agg(
        F.sum("sale_total_price").alias("total_spent"),
        F.round(F.avg("sale_total_price"), 2).alias("avg_check")
    ).orderBy(F.desc("total_spent")).limit(10)

create_table_cust = """
CREATE TABLE IF NOT EXISTS reports.report_customer_sales (
    customer_first_name String,
    customer_last_name String,
    customer_email String,
    customer_country String,
    total_spent Float64,
    avg_check Float64
) ENGINE = MergeTree()
ORDER BY customer_email
"""

try:
    r = requests.post(clickhouse_url, params=auth_params, data=create_table_cust)
    if r.status_code == 200:
        print("report_customer_sales создана успешно.")
    else:
        print(f"Ошибка создания таблицы (Код {r.status_code}): {r.text}")


    if r.status_code == 200:
        for row in report_customers.collect():
            f_name = row.customer_first_name.replace("'", "''")
            l_name = row.customer_last_name.replace("'", "''")
            ins = f"INSERT INTO reports.report_customer_sales VALUES ('{f_name}', '{l_name}', '{row.customer_email}', '{row.customer_country}', {row.total_spent}, {row.avg_check})"
            requests.post(clickhouse_url, params=auth_params, data=ins)
        print("Данные в отчет №2 добавлены.")

except Exception as e:
    print(f"Критическая ошибка, поезд сделал бум: {e}")

✅ Таблица report_customer_sales создана успешно.
✅ Данные в отчет №2 добавлены.


In [14]:
from pyspark.sql import functions as F

report_time = df_fact.withColumn("year", F.year("sale_date")) \
    .withColumn("month", F.month("sale_date")) \
    .groupBy("year", "month") \
    .agg(
        F.sum("sale_total_price").alias("monthly_revenue"),
        F.count("sale_date").alias("total_orders"),
        F.round(F.avg("sale_total_price"), 2).alias("avg_order_value")
    ) \
    .orderBy("year", "month")

data_time = report_time.collect()

create_table_time = """
CREATE TABLE IF NOT EXISTS reports.report_time_sales (
    year Int32,
    month Int32,
    monthly_revenue Float64,
    total_orders Int64,
    avg_order_value Float64
) ENGINE = MergeTree()
ORDER BY (year, month)
"""

try:
    requests.post(clickhouse_url, params=auth_params, data=create_table_time)
    for row in data_time:
        insert_query = f"INSERT INTO reports.report_time_sales VALUES ({row.year}, {row.month}, {row.monthly_revenue}, {row.total_orders}, {row.avg_order_value})"
        requests.post(clickhouse_url, params=auth_params, data=insert_query)
    print("Отчет №3 (Время) успешно записан в ClickHouse! Можно отдохнуть")
    report_time.show(5)
except Exception as e:
    print(f"Ошибка в отчете №3: {e}")

✅ Отчет №3 (Время) успешно записан в ClickHouse!
+----+-----+---------------+------------+---------------+
|year|month|monthly_revenue|total_orders|avg_order_value|
+----+-----+---------------+------------+---------------+
|2021|    1|      224158.54|         874|         256.47|
|2021|    2|      192348.31|         739|         260.28|
|2021|    3|      207282.20|         843|         245.89|
|2021|    4|      206592.82|         837|         246.83|
|2021|    5|      211764.86|         828|         255.75|
+----+-----+---------------+------------+---------------+
only showing top 5 rows



In [18]:
report_stores = df_fact.join(df_stores, "store_id") \
    .groupBy("store_name", "store_city", "store_country") \
    .agg(
        F.sum("sale_total_price").alias("store_revenue"),
        F.round(F.avg("sale_total_price"), 2).alias("avg_check_store")
    ).orderBy(F.desc("store_revenue")).limit(5)

create_table_stores = """
CREATE TABLE IF NOT EXISTS reports.report_store_sales (
    store_name String,
    store_city String,
    store_country String,
    store_revenue Float64,
    avg_check_store Float64
) ENGINE = MergeTree()
ORDER BY store_name
"""

try:
    r = requests.post(clickhouse_url, params=auth_params, data=create_table_stores)
    if r.status_code == 200:
        print("Таблица report_store_sales создана успешно.")
        
        for row in report_stores.collect():
            s_name = row.store_name.replace("'", "''")
            s_city = row.store_city.replace("'", "''")
            ins = f"INSERT INTO reports.report_store_sales VALUES ('{s_name}', '{s_city}', '{row.store_country}', {row.store_revenue}, {row.avg_check_store})"
            requests.post(clickhouse_url, params=auth_params, data=ins)
        print("Данные в отчет №4 добавлены.")
    else:
        print(f"Ошибка создания мироздания: {r.text}")

except Exception as e:
    print(f"Критическая ошибка: {e}")

✅ Таблица report_store_sales создана успешно.
✅ Данные в отчет №4 добавлены.


In [19]:
df_suppliers = spark.read.jdbc(url=db_url, table="dim_suppliers", properties=db_params)
df_products = spark.read.jdbc(url=db_url, table="dim_products", properties=db_params)

df_sup_joined = df_fact.join(df_products, "product_id") \
                       .join(df_suppliers, "supplier_id")

report_suppliers = df_sup_joined.groupBy("supplier_name", "supplier_country") \
    .agg(
        F.sum("sale_total_price").alias("total_supplier_revenue"),
        F.round(F.avg("product_price"), 2).alias("avg_item_price")
    ) \
    .orderBy(F.desc("total_supplier_revenue")) \
    .limit(5) # ТЗ требует Топ-5 (кажется)

create_table_sup = """
CREATE TABLE IF NOT EXISTS reports.report_supplier_sales (
    supplier_name String,
    supplier_country String,
    total_supplier_revenue Float64,
    avg_item_price Float64
) ENGINE = MergeTree()
ORDER BY supplier_name
"""

try:
    r = requests.post(clickhouse_url, params=auth_params, data=create_table_sup)
    if r.status_code == 200:
        print("Таблица report_supplier_sales готова.")
        
        for row in report_suppliers.collect():
            sup_name = row.supplier_name.replace("'", "''")
            ins = f"INSERT INTO reports.report_supplier_sales VALUES ('{sup_name}', '{row.supplier_country}', {row.total_supplier_revenue}, {row.avg_item_price})"
            requests.post(clickhouse_url, params=auth_params, data=ins)
        print("Данные в отчет №5 успешно добавлены.")
    else:
        print(f"Ошибка ClickHouse: {r.text}")

except Exception as e:
    print(f"Ошибка: {e}")

report_suppliers.show()

✅ Таблица report_supplier_sales готова.
✅ Данные в отчет №5 успешно добавлены.
+-------------+----------------+----------------------+--------------+
|supplier_name|supplier_country|total_supplier_revenue|avg_item_price|
+-------------+----------------+----------------------+--------------+
|      Jetwire|       Indonesia|            2529852.12|         31.91|
+-------------+----------------+----------------------+--------------+



In [21]:
report_quality = df.groupBy("product_name", "product_category") \
    .agg(
        F.round(F.avg("product_rating"), 2).alias("avg_rating"),
        F.sum("product_reviews").cast("long").alias("total_reviews"),
        F.sum("sale_quantity").cast("long").alias("total_sales_volume")
    ) \
    .orderBy(F.desc("avg_rating"))

create_table_quality = """
CREATE TABLE IF NOT EXISTS reports.report_product_quality (
    product_name String,
    product_category String,
    avg_rating Float64,
    total_reviews Int64,
    total_sales_volume Int64
) ENGINE = MergeTree()
ORDER BY avg_rating
"""

try:
    r = requests.post(clickhouse_url, params=auth_params, data=create_table_quality)
    if r.status_code == 200:
        print("Таблица report_product_quality создана успешно.")
        
        for row in report_quality.collect():
            p_name = row.product_name.replace("'", "''")
            ins = f"INSERT INTO reports.report_product_quality VALUES ('{p_name}', '{row.product_category}', {row.avg_rating}, {row.total_reviews}, {row.total_sales_volume})"
            requests.post(clickhouse_url, params=auth_params, data=ins)
        print("Все данные в отчет №6 добавлены.")
    else:
        print(f"Ошибка создания таблицы: {r.text}")

except Exception as e:
    print(f"Ошибка: {e}")

report_quality.show(5)

✅ Таблица report_product_quality создана успешно.
✅ Все данные в отчет №6 добавлены.
+------------+----------------+----------+-------------+------------------+
|product_name|product_category|avg_rating|total_reviews|total_sales_volume|
+------------+----------------+----------+-------------+------------------+
|     Cat Toy|             Toy|      3.08|       548517|              6028|
|    Dog Food|            Cage|      3.05|       522231|              5756|
|   Bird Cage|            Food|      3.02|       530547|              5710|
|    Dog Food|             Toy|      3.01|       581510|              6346|
|   Bird Cage|            Cage|      3.00|       584084|              6234|
+------------+----------------+----------+-------------+------------------+
only showing top 5 rows

